# Parsing Request Bodies in JavaScript

How you read a request body depends entirely on the runtime. There are three distinct worlds:

| Environment | Mechanism | Where the data lands |
|---|---|---|
| Express.js | Middleware parses ahead of the handler | `req.body` (synchronous by the time you see it) |
| Fetch / WinterCG (Next.js App Router, Remix, Bun, Deno, Cloudflare Workers) | Promise-based methods on `Request` | `await request.json()` etc. |
| Vanilla Node.js `http` | Manual stream consumption | You assemble it from chunks yourself |

Underneath, all three are doing the same thing: an HTTP body arrives as a **byte stream**, and something has to buffer it and decode it according to `Content-Type`.

---

## 1. Express.js

Express ships body parsers built in since 4.16 — no need to install `body-parser` separately (it's the same code, re-exported).

```javascript
const express = require('express');
const app = express();

// Parse JSON payloads → Content-Type: application/json
app.use(express.json());

// Parse HTML form posts → Content-Type: application/x-www-form-urlencoded
app.use(express.urlencoded({ extended: true }));

app.post('/submit', (req, res) => {
  console.log(req.body);           // already parsed
  res.json({ status: 'Success', receivedData: req.body });
});

app.listen(3000);
```

### The full parser set

| Parser | Handles | Result type |
|---|---|---|
| `express.json()` | `application/json` | Object / Array |
| `express.urlencoded({ extended: true })` | `application/x-www-form-urlencoded` | Object |
| `express.text()` | `text/plain` | String |
| `express.raw()` | `application/octet-stream` | Buffer |
| — | `multipart/form-data` | **Not supported** — use `multer` |

`extended: true` uses `qs`, which supports nested keys (`user[address][city]=Lucknow`). `false` uses Node's `querystring` — flat pairs only.

### Options worth knowing

```javascript
app.use(express.json({
  limit: '1mb',            // default 100kb; oversize → 413 PayloadTooLargeError
  strict: true,            // default: only accept objects/arrays at top level
  type: 'application/vnd.api+json'   // override which Content-Type to parse
}));
```

### Scoping a parser to one route

Global `app.use()` is convenient but blunt. Parsers can be applied per-route:

```javascript
app.post('/submit', express.json(), handler);
```

Useful when one endpoint needs a raw body while the rest of the app wants JSON.

### Handling malformed JSON

If the client sends broken JSON, `express.json()` throws a `SyntaxError` before your handler runs. Catch it explicitly, or the client gets an ugly HTML 400.

```javascript
app.use((err, req, res, next) => {
  if (err instanceof SyntaxError && err.status === 400 && 'body' in err) {
    return res.status(400).json({ error: 'Malformed JSON in request body' });
  }
  next(err);
});
```

### Preserving the raw body (webhook signatures)

Stripe, GitHub, Slack etc. sign the **raw bytes**. Parsing to an object destroys the signature. Either mount `express.raw()` on that route, or stash the buffer with `verify`:

```javascript
// Option A — raw parser on just the webhook route
app.post('/webhook', express.raw({ type: 'application/json' }), (req, res) => {
  verifySignature(req.body, req.headers['stripe-signature']);  // req.body is a Buffer
  const event = JSON.parse(req.body.toString());
  res.sendStatus(200);
});

// Option B — keep both raw and parsed
app.use(express.json({
  verify: (req, res, buf) => { req.rawBody = buf; }
}));
```

### File uploads with `multer`

```javascript
const multer = require('multer');
const upload = multer({ dest: 'uploads/', limits: { fileSize: 5 * 1024 * 1024 } });

app.post('/upload', upload.single('avatar'), (req, res) => {
  console.log(req.file);   // file metadata
  console.log(req.body);   // any accompanying text fields
  res.sendStatus(201);
});
```

`upload.array('photos', 10)` for multiple, `upload.fields([...])` for mixed. `formidable` and `busboy` are alternatives.

---

## 2. Fetch / Web-Standard Runtimes

Next.js App Router, Remix, Bun, Deno, Cloudflare Workers, and Node 18+ all implement the standard `Request` interface. Parsing is promise-based because the body is a stream being consumed.

```javascript
async function handleRequest(request) {
  const contentType = request.headers.get('content-type') ?? '';

  if (contentType.includes('application/json')) {
    const body = await request.json();
    return Response.json(body);
  }

  if (contentType.includes('application/x-www-form-urlencoded') ||
      contentType.includes('multipart/form-data')) {
    const formData = await request.formData();
    return Response.json(Object.fromEntries(formData));
  }

  return new Response('Unsupported Media Type', { status: 415 });
}
```

### Methods on `Request` (and `Response`)

| Method | Returns |
|---|---|
| `await request.json()` | Parsed object |
| `await request.text()` | String |
| `await request.formData()` | `FormData` — handles both urlencoded *and* multipart, including files |
| `await request.arrayBuffer()` | `ArrayBuffer` |
| `await request.blob()` | `Blob` |
| `request.body` | `ReadableStream` — for manual/streaming consumption |

`formData()` is the quiet win here: unlike Express, the web standard handles file uploads with no extra library. Files come back as `File` objects.

### The body can only be read once

Bodies are single-use streams. A second call throws.

```javascript
const a = await request.json();
const b = await request.json();   // TypeError: Body is unusable

// Need it twice (e.g. log the raw text, then parse)? Clone first:
const raw = await request.clone().text();
const parsed = await request.json();
```

Check `request.bodyUsed` if you're unsure of state.

### Next.js specifics

- **App Router** (`app/api/route.js`) — standard `Request`, use the methods above.
- **Pages Router** (`pages/api/*.js`) — Next parses for you; read `req.body` like Express. Disable via `export const config = { api: { bodyParser: false } }` when you need the raw stream.

### Content-Type matching

Always use `.includes()`, never `===`. Real headers carry parameters:

```
application/json; charset=utf-8
multipart/form-data; boundary=----WebKitFormBoundary7MA4YW
```

---

## 3. Vanilla Node.js

No framework, no magic — the request *is* a readable stream, and you buffer it yourself.

```javascript
const http = require('http');

const MAX_BODY = 1e6;   // 1 MB guard

const server = http.createServer((req, res) => {
  if (req.method !== 'POST') {
    res.writeHead(405).end();
    return;
  }

  const chunks = [];
  let size = 0;

  req.on('data', chunk => {
    size += chunk.length;
    if (size > MAX_BODY) {
      res.writeHead(413).end('Payload too large');
      req.destroy();               // stop reading — critical
      return;
    }
    chunks.push(chunk);
  });

  req.on('end', () => {
    const rawBodyText = Buffer.concat(chunks).toString('utf8');
    try {
      const parsedBody = JSON.parse(rawBodyText);
      res.writeHead(200, { 'Content-Type': 'application/json' });
      res.end(JSON.stringify({ message: 'Parsed successfully!', data: parsedBody }));
    } catch {
      res.writeHead(400, { 'Content-Type': 'text/plain' });
      res.end('Invalid JSON format');
    }
  });

  req.on('error', err => {
    console.error(err);
    res.writeHead(500).end();
  });
});

server.listen(3000);
```

> **The size guard is not optional.** Without it, an attacker streams an unbounded body and exhausts server memory. This is exactly what `express.json({ limit })` does for you.

### Shortcut: `node:stream/consumers`

Node 16.7+ has helpers that collapse the boilerplate:

```javascript
const { json, text, buffer } = require('node:stream/consumers');

const server = http.createServer(async (req, res) => {
  try {
    const body = await json(req);      // buffers + parses in one line
    res.end(JSON.stringify(body));
  } catch {
    res.writeHead(400).end('Bad JSON');
  }
});
```

No built-in size limit though — still your job.

### Parsing urlencoded manually

```javascript
const params = new URLSearchParams(rawBodyText);
const data = Object.fromEntries(params);
```

---

## Sending a Body (client side)

Worth keeping next to the parsing notes, since mismatches here cause most "why is `req.body` empty" bugs.

```javascript
// JSON
await fetch('/submit', {
  method: 'POST',
  headers: { 'Content-Type': 'application/json' },
  body: JSON.stringify({ name: 'Harshit' })
});

// Form data — do NOT set Content-Type manually;
// the browser adds it with the correct multipart boundary
const fd = new FormData();
fd.append('name', 'Harshit');
fd.append('file', fileInput.files[0]);
await fetch('/upload', { method: 'POST', body: fd });

// URL-encoded
await fetch('/submit', {
  method: 'POST',
  headers: { 'Content-Type': 'application/x-www-form-urlencoded' },
  body: new URLSearchParams({ name: 'Harshit' })
});
```

---

## Validation — Never Trust the Body

Parsing tells you the shape is *syntactically* valid JSON. It says nothing about whether the fields are what you expect. Validate at the boundary, then work with typed data.

```javascript
const { z } = require('zod');

const UserSchema = z.object({
  name: z.string().min(1).max(100),
  email: z.string().email(),
  age: z.number().int().positive().optional()
});

app.post('/users', (req, res) => {
  const result = UserSchema.safeParse(req.body);

  if (!result.success) {
    return res.status(422).json({ errors: result.error.flatten() });
  }

  const user = result.data;   // validated + typed, extra keys stripped
  res.status(201).json(user);
});
```

Alternatives: **Joi**, **Yup**, **express-validator**, **Valibot**, **class-validator**. Zod is the current default in TS-heavy stacks because the schema doubles as the type.

**Why it matters:** unvalidated body properties feed straight into SQL injection, NoSQL operator injection (`{ "$gt": "" }` as a password), prototype pollution (`__proto__` keys), and mass-assignment bugs (client sends `{ role: "admin" }` and your ORM happily writes it).

---

## Common Gotchas

- **`req.body` is `undefined`** → parser middleware missing, or registered *after* the route.
- **`req.body` is `{}`** → the client didn't send a matching `Content-Type`, so the parser skipped the body silently.
- **Middleware order** — `app.use()` runs top-to-bottom; anything registered after a matching route never fires for it.
- **GET/DELETE bodies** — parsers only run when a body is present; most clients and proxies won't send one. Don't design around it.
- **Reading a Fetch body twice** → `TypeError`. Use `.clone()`.
- **Parsing before signature verification** breaks webhooks irreversibly.
- **`multipart/form-data` in Express** silently yields `{}` without `multer` — the built-in parsers don't touch it.
- **Charset** — Express assumes UTF-8; exotic encodings need explicit handling.
- **Empty body with `express.json()`** → yields `{}`, not an error. Check for required fields, don't assume a throw.

---

## Related

- [[Express POST Routes]]
- [[Express Middleware]]
- [[HTTP Status Codes]]
- [[Fetch API]]
- [[Node.js Streams]]

## References

- [MDN — `Request.json()`](https://developer.mozilla.org/en-US/docs/Web/API/Request/json)
- [Express — `express.json()` API](https://expressjs.com/en/api.html#express.json)
- [npm — body-parser](https://www.npmjs.com/package/body-parser)
- [npm — multer](https://www.npmjs.com/package/multer)
- [Zod documentation](https://zod.dev)